# Feature engineering — the region-season behavioral fingerprint

This notebook builds the **cross-sectional feature table** the learned models in
[`06_analysis.ipynb`](06_analysis.ipynb) train on. It sits in the pipeline after cleaning and after the
analysis grain, mirroring the `04_cleaning.ipynb → region_season_cause.parquet` seam:

- **Input** — `data/region_season_cause.parquet` (EPA Level III ecoregion × season-year × cause).
- **Output** — `data/region_season_features.parquet`: one row per **region-season cell**
  (region × season_idx), carrying trailing "fingerprint" features **plus** the Tier-1 targets
  (the three composition shares and the log total-acres level), so the model notebook loads one
  self-contained table.

**Why a separate notebook.** The persistence baselines showed the target is *static* — there is
little season-to-season signal — so the next rung must find lift **cross-sectionally**: predict a
region-season from *what kind of region-season it is*, letting similar cells inform each other,
rather than from its own timeline alone. Every feature here is therefore a **trailing summary of
strictly-prior same-region/same-season cells** — the exact forward-chaining discipline the
baselines use (`shift(1)` then aggregate, so the target year is never in its own features). This
is the highest-leakage-risk code in the project, so it is quarantined in one auditable artifact
with an explicit leakage check before anything is written.

**Notebook conventions.** Cells run top-to-bottom and are left **unexecuted** for the student to
run manually, following project practice.


## Load the grain and rebuild the per-cell targets

The targets are reconstructed exactly as in `06_analysis.ipynb` Tier 1: coarse Human/Natural/Unknown
acres on a *total-acres* denominator (resolved + `missing_acres`), giving the three shares (which
sum to 1) and the log total-acres level. Fire **counts** are carried too, for the regime-shape
features below.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from config import ProjectConfig

# Project-wide constants (paths, the boundary rule, the trailing window) come from one
# place so this table cannot drift from the baselines it is compared against.
cfg = ProjectConfig()
IN_PATH = cfg.region_season_cause
OUT_PATH = cfg.region_season_features

rsc = pd.read_parquet(IN_PATH)
rsc["coarse"] = rsc["cause"].map(lambda c: "Natural" if c == "Natural" else "Human")

# Resolved acres by coarse class, one row per cell.
resolved = (rsc.groupby(["region", "season", "season_idx", "season_year", "coarse"], observed=True)["acres"]
            .sum().unstack("coarse", fill_value=0.0)
            .rename(columns={"Human": "human_ac", "Natural": "natural_ac"}).reset_index())

# Unknown acres/fires: one value per cell (constant within a cell; dedup, don't sum).
unk = rsc.drop_duplicates(["region", "season_idx"])[["region", "season_idx", "missing_acres", "missing_fires"]]
cell = resolved.merge(unk, on=["region", "season_idx"], how="left").rename(
    columns={"missing_acres": "unknown_ac", "missing_fires": "unknown_fires"})
cell["unknown_fires"] = cell["unknown_fires"].fillna(0.0)

# Resolved fire counts per cell (for regime-shape features).
rfires = rsc.groupby(["region", "season_idx"], observed=True)["fires"].sum().rename("resolved_fires")
cell = cell.merge(rfires, on=["region", "season_idx"], how="left")

cell["total_ac"] = cell[["human_ac", "natural_ac", "unknown_ac"]].sum(axis=1)
cell["total_fires"] = cell["resolved_fires"] + cell["unknown_fires"]
cell = cell[cell["total_ac"] > 0].copy()                 # a cell needs some burned area

for c, ac in zip(cfg.tier1_classes, ["human_ac", "natural_ac", "unknown_ac"]):
    cell[c] = cell[ac] / cell["total_ac"]                 # TARGET shares (sum to 1)
cell["log_total"] = np.log10(cell["total_ac"])            # TARGET level (log10 acres)
cell["mean_fire_size"] = cell["total_ac"] / cell["total_fires"].clip(lower=1)

cell = cell.sort_values(list(cfg.sort_keys)).reset_index(drop=True)
assert (cell[list(cfg.tier1_classes)].sum(axis=1).sub(1).abs() < 1e-9).all()
print(f"{len(cell):,} region-season cells | season_years {cell.season_year.min()}-{cell.season_year.max()}")
cell.head()

In [ ]:
# Boundary rule — drop the two partial winters (season_idx 0 and 116), the DJF seasons truncated
# by the record's 1992-01-01 / 2020-12-31 endpoints. The rule lives in src/config.py so this table
# and the baselines in 06_analysis.ipynb are built on an identical set of region-season cells.
PARTIAL_WINTERS = list(cfg.partial_winters)
_before = len(cell)
cell = cell[~cell["season_idx"].isin(PARTIAL_WINTERS)].copy()
cell = cell.sort_values(list(cfg.sort_keys)).reset_index(drop=True)
print(f"dropped partial winters {PARTIAL_WINTERS}: {_before - len(cell)} cells removed "
      f"({_before:,} -> {len(cell):,})")

## Build the trailing fingerprint features

Each feature summarizes the **last `K` strictly-prior same-season occurrences** of the same region.
The pattern is always `shift(1)` (drop the target year) then a rolling aggregate within
`(region, season)` — identical to the persistence baselines, so a feature for `season_idx = t`
can only ever see cells at earlier same-season years. `K = 7` matches the shares baseline window.

The fingerprint answers *what kind of region-season is this?* along five axes:

- **Typical size** (`f_log_total_mean`) — the trailing geometric-mean burn; this is literally the
  level baseline, now available as a feature.
- **Volatility** (`f_log_total_std`) — spiky (Yukon Flats) vs steady (Arizona/NM).
- **Typical composition** (`f_share_{human,natural,unknown}_mean`) — the shares baseline as features.
- **Regime shape** (`f_log_fire_size_mean`, `f_log_nfires_mean`) — few-big vs many-small.
- **History depth** (`f_n_prior`) — how many prior same-season years the fingerprint rests on; a
  confidence signal (thin history → less reliable features).

Cells with no prior history (a region-season's first occurrence) get **NaN** features — the same
cells the baselines leave unscored.

In [ ]:
K = cfg.shares_k   # trailing window (matches the shares baseline)

# The forward-chaining rule lives in src/trailing.py rather than being re-typed here.
# TrailingMean does shift(1) then a k-window aggregate within (region, season), and
# asserts the frame is sorted by (region, season, season_idx) first -- on an unsorted
# frame the raw shift-then-roll idiom silently attaches one region's history to
# another's rows, with no NaN and no error. This is the highest-leakage-risk code in
# the project, so the guarantee is enforced by the class, not by call-site discipline.
from trailing import TrailingMean

mean_k = TrailingMean(K)                                  # min_periods=1
std_k = TrailingMean(K, min_periods=2, how="std")          # spread needs >=2 prior years

feat = cell[["region", "season", "season_idx", "season_year"]].copy()

# level & volatility
feat["f_log_total_mean"] = mean_k.predict(cell, "log_total")["log_total"]
feat["f_log_total_std"]  = std_k.predict(cell, "log_total")["log_total"]

# composition (shares baseline, as features)
for c in cfg.tier1_classes:
    feat[f"f_share_{c}_mean"] = mean_k.predict(cell, c)[c]

# regime shape (log-scaled: both are heavy-tailed)
feat["f_log_fire_size_mean"] = np.log10(
    mean_k.predict(cell, "mean_fire_size")["mean_fire_size"].clip(lower=1e-6))
feat["f_log_nfires_mean"] = np.log10(
    mean_k.predict(cell, "total_fires")["total_fires"].clip(lower=1))

# history depth -- number of prior same-season years the fingerprint rests on
feat["f_n_prior"] = cell.groupby(["region", "season"], observed=True).cumcount()

FEATCOLS = [c for c in feat.columns if c.startswith("f_")]
print(f"{len(FEATCOLS)} features: {FEATCOLS}")
feat[FEATCOLS].describe().round(3).T

## Attach the targets

Carry the Tier-1 targets onto the same rows so the model notebook loads a single table: the three
composition shares, the log-level, and raw `total_ac` (the acre weight the scoring uses).

In [4]:
for t in ["human", "natural", "unknown", "log_total", "total_ac"]:
    feat[t] = cell[t].to_numpy()

# season is a legitimate (non-leaking) identity feature; keep it explicit for the model's one-hot.
print(feat[["human", "natural", "unknown"]].describe().round(3).to_string())
feat.head()

           human    natural    unknown
count  10135.000  10135.000  10135.000
mean       0.592      0.170      0.238
std        0.352      0.308      0.280
min        0.000      0.000      0.000
25%        0.269      0.000      0.014
50%        0.678      0.005      0.118
75%        0.924      0.150      0.376
max        1.000      1.000      1.000


,region,season,season_idx,season_year,f_log_total_mean,f_log_total_std,f_share_human_mean,f_share_natural_mean,f_share_unknown_mean,f_log_fire_size_mean,f_log_nfires_mean,f_n_prior,human,natural,unknown,log_total,total_ac
0,Acadian Plains and Hills,DJF,8,1994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1.000000,0.0,0.000000,-0.698970,0.20
1,Acadian Plains and Hills,DJF,32,2000,-0.698970,NaN,1.000000,0.0,0.000000,-1.000000,0.301030,1,0.165289,0.0,0.834711,0.082785,1.21
2,Acadian Plains and Hills,DJF,40,2002,-0.308092,0.552785,0.582645,0.0,0.417355,-0.900924,0.698970,2,1.000000,0.0,0.000000,0.086360,1.22
3,Acadian Plains and Hills,DJF,48,2004,-0.176608,0.452382,0.721763,0.0,0.278237,-0.918222,0.845098,3,1.000000,0.0,0.000000,-0.301030,0.50
4,Acadian Plains and Hills,DJF,56,2006,-0.207714,0.374571,0.791322,0.0,0.208678,-0.666473,0.740363,4,0.800766,0.0,0.199234,1.115611,13.05


## Leakage audit

Before writing, independently verify the forward-chaining guarantee: for a random sample of cells,
reconstruct `f_log_total_mean` from scratch out of the strictly-earlier same-season cells and
confirm it matches the vectorized feature. A single mismatch means a feature saw the future — a
hard stop. The first-occurrence cells (no history) must be exactly the NaN-feature cells.

In [5]:
# Brute-force recompute f_log_total_mean for a sample and compare to the vectorized column.
sample = feat[feat["f_n_prior"] > 0].sample(min(500, int((feat["f_n_prior"] > 0).sum())), random_state=1)
mismatches = 0
for _, row in sample.iterrows():
    prior = cell[(cell.region == row.region) & (cell.season == row.season)
                 & (cell.season_idx < row.season_idx)].sort_values("season_idx").tail(K)
    expected = prior["log_total"].mean()
    if not np.isclose(row["f_log_total_mean"], expected, rtol=1e-6, atol=1e-6):
        mismatches += 1
assert mismatches == 0, f"LEAKAGE: {mismatches} features did not match a strictly-prior recompute"
print(f"leakage audit passed: 0 mismatches over {len(sample)} sampled cells")

# First-occurrence cells (no prior history) are exactly the ones with NaN features.
no_hist = feat["f_n_prior"] == 0
nan_feat = feat[FEATCOLS].isna().any(axis=1)
assert (no_hist == (feat["f_log_total_mean"].isna())).all(), "history-depth and NaN features disagree"
print(f"{no_hist.sum():,} first-occurrence cells carry NaN features (unscored, as in the baselines)")

leakage audit passed: 0 mismatches over 500 sampled cells
402 first-occurrence cells carry NaN features (unscored, as in the baselines)


## Write `region_season_features.parquet`

One row per region-season cell: identity keys, the trailing fingerprint features, and the Tier-1
targets. `06_analysis.ipynb` loads this read-only for the cross-sectional model rungs, scored against
the same two floors (composition total variation distance (TVD) ≈ 0.27, level ≈ 9× off) on the same
forward-chaining split.

In [6]:
feat.to_parquet(OUT_PATH, index=False)
print(f"wrote {OUT_PATH}  ({len(feat):,} rows x {feat.shape[1]} cols)")
print("columns:", list(feat.columns))

wrote ../data/region_season_features.parquet  (10,135 rows x 17 cols)
columns: ['region', 'season', 'season_idx', 'season_year', 'f_log_total_mean', 'f_log_total_std', 'f_share_human_mean', 'f_share_natural_mean', 'f_share_unknown_mean', 'f_log_fire_size_mean', 'f_log_nfires_mean', 'f_n_prior', 'human', 'natural', 'unknown', 'log_total', 'total_ac']
